In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder\
        .master('local[*]')\
        .appName('nti_project')\
        .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [3]:
def read_data(base_path , table_name):
    df = spark.read.parquet(f'{base_path}{table_name}')
    return df
    
def clean_dataframe(df):
    return df.select([ F.initcap(F.trim(F.col(c))).alias(c) for c in df.columns ])

def check_duplicates(df, key_column):
    duplicates = df.groupBy(key_column).count().filter(F.col('count') > 1)
    return duplicates

def remove_duplicates(df , key_column):
    print('Before Remove Duplicates' , df.count())
    df = df.dropDuplicates([key_column])
    print('After Remove Duplicates' , df.count())
    return df

def count_nulls(df):
    df_columns = df.columns

    for column in df_columns:
        null_count = df.filter(F.col(column).isNull()).count()
        print(f"{column}: {null_count} null values")

def remove_columns(df , *columns):
    print('Before Drop Columns' , len(df.columns))
    df = df.drop(*columns)
    print('After Drop Columns' , len(df.columns))
    return df

In [4]:
base_path = 'hdfs://hadoop-namenode:9000/Data/parquet/'

### Categories Cleaning

In [5]:
Categories = read_data(base_path , 'Categories')

In [6]:
Categories.printSchema()

root
 |-- CategoryID: string (nullable = true)
 |-- CategoryName: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Picture: string (nullable = true)



In [7]:
duplicate_ids = count_nulls(Categories)

CategoryID: 0 null values
CategoryName: 0 null values
Description: 0 null values
Picture: 8 null values


In [8]:
result = check_duplicates(Categories , 'CategoryID')
result.show()

+----------+-----+
|CategoryID|count|
+----------+-----+
+----------+-----+



In [9]:
remove_column = ['Picture']
Categories = remove_columns(Categories , *remove_column)

Before Drop Columns 4
After Drop Columns 3


In [10]:
Categories = clean_dataframe(Categories)

In [11]:
dim_categories = Categories  

dim_categories.write.mode('overwrite').parquet(
    "hdfs://hadoop-namenode:9000/Data/silver/dim_categories"
)

### Customers Cleaning

In [12]:
Customers = read_data(base_path , 'Customers')

In [13]:
Customers.printSchema()

root
 |-- CustomerID: string (nullable = true)
 |-- CompanyName: string (nullable = true)
 |-- ContactName: string (nullable = true)
 |-- ContactTitle: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- PostalCode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Fax: string (nullable = true)



In [14]:
duplicate_ids = count_nulls(Customers)

CustomerID: 0 null values
CompanyName: 0 null values
ContactName: 0 null values
ContactTitle: 0 null values
Address: 0 null values
City: 0 null values
Region: 60 null values
PostalCode: 1 null values
Country: 0 null values
Phone: 0 null values
Fax: 22 null values


In [15]:
result = check_duplicates(Customers , 'CustomerID')
result.show()

+----------+-----+
|CustomerID|count|
+----------+-----+
+----------+-----+



In [16]:
result = remove_duplicates(Customers , 'CustomerID')
result

Before Remove Duplicates 91
After Remove Duplicates 91


DataFrame[CustomerID: string, CompanyName: string, ContactName: string, ContactTitle: string, Address: string, City: string, Region: string, PostalCode: string, Country: string, Phone: string, Fax: string]

In [17]:
remove_column = ['Region','PostalCode','Fax']
Customers = remove_columns(Customers , *remove_column)

Before Drop Columns 11
After Drop Columns 8


In [18]:
Customers = clean_dataframe(Customers)

In [19]:
dim_customers = Customers  

dim_customers.write.mode('overwrite').parquet(
    "hdfs://hadoop-namenode:9000/Data/silver/dim_customers"
)

### Orders Cleaning

In [20]:
Orders = read_data(base_path , 'Orders')

In [21]:
Orders.printSchema()

root
 |-- OrderID: string (nullable = true)
 |-- Customer key : string (nullable = true)
 |-- EmployeeID: string (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- RequiredDate: string (nullable = true)
 |-- ShippedDate: string (nullable = true)
 |-- ShipVia: string (nullable = true)
 |-- Freight: string (nullable = true)
 |-- ShipName: string (nullable = true)
 |-- ShipAddress: string (nullable = true)
 |-- ShipCity: string (nullable = true)
 |-- ShipRegion: string (nullable = true)
 |-- ShipPostalCode: string (nullable = true)
 |-- ShipCountry: string (nullable = true)
 |-- Column1: string (nullable = true)
 |-- Column2: string (nullable = true)
 |-- Column3: string (nullable = true)
 |-- Column4: string (nullable = true)
 |-- Column5: string (nullable = true)
 |-- Column6: string (nullable = true)
 |-- Column7: string (nullable = true)
 |-- Column8: string (nullable = true)
 |-- Column9: string (nullable = true)
 |-- Column10: string (nullable = true)
 |-- Column11: str

In [22]:
duplicate_ids = count_nulls(Orders)

OrderID: 0 null values
Customer key : 0 null values
EmployeeID: 0 null values
OrderDate: 0 null values
RequiredDate: 0 null values
ShippedDate: 21 null values
ShipVia: 0 null values
Freight: 0 null values
ShipName: 0 null values
ShipAddress: 0 null values
ShipCity: 0 null values
ShipRegion: 510 null values
ShipPostalCode: 19 null values
ShipCountry: 0 null values
Column1: 834 null values
Column2: 834 null values
Column3: 834 null values
Column4: 834 null values
Column5: 834 null values
Column6: 834 null values
Column7: 834 null values
Column8: 834 null values
Column9: 834 null values
Column10: 834 null values
Column11: 834 null values


In [23]:
result = check_duplicates(Orders , 'OrderID')
result.show()

+-------+-----+
|OrderID|count|
+-------+-----+
|  10251|    2|
|  10253|    2|
|  10252|    2|
|  10254|    2|
+-------+-----+



In [24]:
Orders = remove_duplicates(Orders , 'OrderID')

Before Remove Duplicates 834
After Remove Duplicates 830


In [25]:
columns = ['ShipRegion','ShipPostalCode']
nums = list(range(1,12))
for num in nums:
    column = f'Column{num}'
    columns.append(column)
Orders = remove_columns(Orders , *columns)

Before Drop Columns 25
After Drop Columns 12


In [26]:
Orders = Orders.withColumnRenamed('Customer key ', 'CustomerID')

In [27]:
dim_orders = Orders  

dim_orders.write.mode('overwrite').parquet(
    "hdfs://hadoop-namenode:9000/Data/silver/dim_orders"
)

### OrdersDetails Cleaning

In [28]:
OrdersDetails = read_data(base_path , 'OrdersDetails')

In [29]:
OrdersDetails.printSchema()

root
 |-- OrderID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)



In [30]:
duplicate_ids = count_nulls(OrdersDetails)

OrderID: 0 null values
ProductID: 0 null values
UnitPrice: 0 null values
Quantity: 0 null values
Discount: 0 null values


In [31]:
result = check_duplicates(OrdersDetails , 'OrderID')
result.show()

+-------+-----+
|OrderID|count|
+-------+-----+
|  10351|    8|
|  10436|    8|
|  10272|    6|
|  10309|   10|
|  10338|    4|
|  10603|    4|
|  10250|    6|
|  10730|    6|
|  10784|    6|
|  10801|    4|
|  10296|    6|
|  10616|    8|
|  10831|    8|
|  10993|    4|
|  10417|    8|
|  10481|    4|
|  10668|    6|
|  10805|    4|
|  10872|    8|
|  11002|    8|
+-------+-----+
only showing top 20 rows



In [32]:
print('Before Remove Duplicates' , OrdersDetails.count())
OrdersDetails = OrdersDetails.dropDuplicates()
print('After Remove Duplicates' , OrdersDetails.count())

Before Remove Duplicates 4309
After Remove Duplicates 2155


In [33]:
duplicate_ids = check_duplicates(OrdersDetails , 'OrderID')
duplicate_ids.show()

+-------+-----+
|OrderID|count|
+-------+-----+
|  10436|    4|
|  10351|    4|
|  10309|    5|
|  10272|    3|
|  10603|    2|
|  10338|    2|
|  10784|    3|
|  10730|    3|
|  10801|    2|
|  10250|    3|
|  10296|    3|
|  10616|    4|
|  10831|    4|
|  10993|    2|
|  10668|    3|
|  11002|    4|
|  10481|    2|
|  10417|    4|
|  10872|    4|
|  10805|    2|
+-------+-----+
only showing top 20 rows



In [34]:
OrdersDetails = clean_dataframe(OrdersDetails)

In [35]:
OrdersDetails = OrdersDetails.withColumn('Total_price' , F.round(F.col("UnitPrice") * F.col("Quantity") * (1 - F.col("Discount")), 2) )

In [36]:
dim_ordersdetails = OrdersDetails  

dim_ordersdetails.write.mode('overwrite').parquet(
    "hdfs://hadoop-namenode:9000/Data/silver/dim_ordersdetails"
)

### Product Cleaning

In [37]:
Product = read_data(base_path , 'Product')

In [38]:
Product.printSchema()

root
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- SupplierID: string (nullable = true)
 |-- CategoryID: string (nullable = true)
 |-- QuantityPerUnit: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- UnitsInStock: string (nullable = true)
 |-- UnitsOnOrder: string (nullable = true)
 |-- ReorderLevel: string (nullable = true)
 |-- Discontinued: string (nullable = true)



In [39]:
duplicate_ids = count_nulls(Product)

ProductID: 0 null values
ProductName: 0 null values
SupplierID: 0 null values
CategoryID: 0 null values
QuantityPerUnit: 0 null values
UnitPrice: 0 null values
UnitsInStock: 0 null values
UnitsOnOrder: 0 null values
ReorderLevel: 0 null values
Discontinued: 0 null values


In [40]:
result = check_duplicates(Product , 'ProductID')
result.show()

+---------+-----+
|ProductID|count|
+---------+-----+
+---------+-----+



In [41]:
Product = clean_dataframe(Product)

In [42]:
Product = Product.withColumn('Discontinued' , F.when(F.col('Discontinued')==False , 0)
                                              .when(F.col('Discontinued')==True , 1) )

In [43]:
dim_product = Product  

dim_product.write.mode('overwrite').parquet(
    "hdfs://hadoop-namenode:9000/Data/silver/dim_product"
)

### Employees Table

In [44]:
Employees = read_data(base_path , 'Employees')

In [45]:
Employees.printSchema()

root
 |-- EmployeeID: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- TitleOfCourtesy: string (nullable = true)
 |-- BirthDate: string (nullable = true)
 |-- HireDate: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- PostalCode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- HomePhone: string (nullable = true)
 |-- Extension: string (nullable = true)
 |-- Photo: string (nullable = true)
 |-- Notes: string (nullable = true)
 |-- ReportsTo: string (nullable = true)



In [46]:
duplicate_ids = count_nulls(Employees)

EmployeeID: 0 null values
LastName: 0 null values
FirstName: 0 null values
Title: 0 null values
TitleOfCourtesy: 0 null values
BirthDate: 0 null values
HireDate: 0 null values
Address: 0 null values
City: 0 null values
Region: 4 null values
PostalCode: 0 null values
Country: 0 null values
HomePhone: 0 null values
Extension: 0 null values
Photo: 0 null values
Notes: 0 null values
ReportsTo: 1 null values


In [47]:
result = check_duplicates(Employees , 'EmployeeID')
result.show()

+----------+-----+
|EmployeeID|count|
+----------+-----+
+----------+-----+



In [48]:
Employees = Employees.withColumn('Full_Name' , F.concat_ws(" ",F.trim(F.col("FirstName")), F.trim(F.col("LastName"))))

In [49]:
columns = ['ReportsTo' , 'Notes' , 'Photo' , 'PostalCode' , 'Region' , 'FirstName' , 'LastName']

Employees = remove_columns(Employees , *columns)

Before Drop Columns 18
After Drop Columns 11


In [50]:
Employees = clean_dataframe(Employees)

In [51]:
dim_employees = Employees  

dim_employees.write.mode('overwrite').parquet(
    "hdfs://hadoop-namenode:9000/Data/silver/dim_employees"
)

### Suppliers Cleaning

In [52]:
Suppliers = read_data(base_path , 'Suppliers')

In [53]:
Suppliers.printSchema()

root
 |-- SupplierID: string (nullable = true)
 |-- CompanyName: string (nullable = true)
 |-- ContactName: string (nullable = true)
 |-- ContactTitle: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- PostalCode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Fax: string (nullable = true)
 |-- HomePage: string (nullable = true)



In [54]:
duplicate_ids = count_nulls(Suppliers)

SupplierID: 0 null values
CompanyName: 0 null values
ContactName: 0 null values
ContactTitle: 0 null values
Address: 0 null values
City: 0 null values
Region: 20 null values
PostalCode: 0 null values
Country: 0 null values
Phone: 0 null values
Fax: 16 null values
HomePage: 26 null values


In [55]:
result = check_duplicates(Suppliers , 'SupplierID')
result.show()

+----------+-----+
|SupplierID|count|
+----------+-----+
+----------+-----+



In [56]:
columns = ['Region' , 'PostalCode' , 'Fax' , 'HomePage']

Suppliers = remove_columns(Suppliers , *columns)

Before Drop Columns 12
After Drop Columns 8


In [57]:
Suppliers = clean_dataframe(Suppliers)

In [58]:
dim_suppliers = Suppliers  

dim_suppliers.write.mode('overwrite').parquet(
    "hdfs://hadoop-namenode:9000/Data/silver/dim_suppliers"
)

### Shippers Cleaning

In [59]:
Shippers = read_data(base_path , 'Shippers')

In [60]:
Shippers.printSchema()

root
 |-- ShipperID: string (nullable = true)
 |-- CompanyName: string (nullable = true)
 |-- Phone: string (nullable = true)



In [61]:
duplicate_ids = count_nulls(Shippers)

ShipperID: 0 null values
CompanyName: 0 null values
Phone: 0 null values


In [62]:
result = check_duplicates(Shippers , 'ShipperID')
result.show()

+---------+-----+
|ShipperID|count|
+---------+-----+
+---------+-----+



In [63]:
Shippers = clean_dataframe(Shippers)

In [64]:
dim_shippers = Shippers  

dim_shippers.write.mode('overwrite').parquet(
    "hdfs://hadoop-namenode:9000/Data/silver/dim_shippers"
)